# Project 3 - Learning-Enhanced Prediction

This walkthrough fits a least-squares residual model for one-step prediction. The teaching goal is not to sell learning as a safety guarantee; it is to show where a learned correction can improve prediction and where validation is still required.


In [ ]:
%matplotlib inline
from pathlib import Path
import os
import sys
import numpy as np
from IPython.display import FileLink, display

repo_root = Path.cwd()
while not (repo_root / "projects").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from projects.project_3_learning_enhanced_prediction import config, scenario
from projects.project_3_learning_enhanced_prediction.animation import save_prediction_replay_html
from projects.project_3_learning_enhanced_prediction.plots import plot_prediction_eta, plot_residual_error, plot_rmse
from systems.mobile_robot import tracking_error_matrices
from systems.residual_models import fit_residual_least_squares, predict_residual

output_root = Path(os.environ.get("THIMPC_OUTPUT_DIR", "/tmp/thimpc_walkthroughs"))
project_output = output_root / "project_3_learning_enhanced_prediction"
np.set_printoptions(precision=3, suppress=True)


## Main Experiment Parameters

The split and noise setting are visible because they strongly affect what the validation result means.


In [ ]:
dt = config.DT
v_ref = config.V_REF
omega_ref = config.OMEGA_REF
samples = int(os.environ.get("THIMPC_PROJECT3_SAMPLES", "160"))
seed = config.SEED
train_fraction = 0.7
ridge = 1e-6
measurement_noise_std = 0.0

print(f"dt = {dt}, reference speed = {v_ref}, reference yaw rate = {omega_ref}")
print(f"samples = {samples}, seed = {seed}, train fraction = {train_fraction}")
print(f"measurement noise std = {measurement_noise_std}, ridge = {ridge}")


## Data-Generation Model

The nominal prediction is linear in the tracking error and yaw-rate correction:

`e_nom[k+1] = A_e e[k] + B_e delta_omega[k]`

The measured transition adds a small systematic residual. We will fit that residual from data.


In [ ]:
A_error, B_error = tracking_error_matrices(dt, v_ref, omega_ref)
errors, inputs = scenario.generate_data(samples, seed=seed)
measured_next = scenario.true_transition(errors, inputs, A_error, B_error)

if measurement_noise_std > 0.0:
    rng = np.random.default_rng(seed + 1)
    measured_next = measured_next + rng.normal(0.0, measurement_noise_std, measured_next.shape)

nominal_next = errors @ A_error.T + inputs.reshape(-1, 1) @ B_error.T
residual = measured_next - nominal_next

print("A_error =\n", A_error)
print("B_error =\n", B_error)
print("errors shape =", errors.shape)
print("inputs shape =", inputs.shape)
print("residual shape =", residual.shape)


## Train/Validation Split

The model is fitted only on the training slice. The validation slice is held out so the comparison is not just memorization.


In [ ]:
split = max(20, int(train_fraction * samples))
train = slice(0, split)
validation = slice(split, samples)

print("training samples =", split)
print("validation samples =", samples - split)


## Least-Squares Residual Fit

We fit a map from simple features of `(error, input)` to the residual:

`measured_next - nominal_next`.

The helper hides only the feature-matrix bookkeeping; the idea is ordinary regularized least squares.


In [ ]:
W = fit_residual_least_squares(errors[train], inputs[train], residual[train], ridge=ridge)
learned_residual_validation = predict_residual(W, errors[validation], inputs[validation])
learned_next_validation = nominal_next[validation] + learned_residual_validation

nominal_error = measured_next[validation] - nominal_next[validation]
learned_error = measured_next[validation] - learned_next_validation
nominal_rmse = np.sqrt(np.mean(nominal_error * nominal_error, axis=0))
learned_rmse = np.sqrt(np.mean(learned_error * learned_error, axis=0))

print("residual weight matrix W shape =", W.shape)
print("nominal RMSE [xi, eta, psi] =", nominal_rmse)
print("learned RMSE [xi, eta, psi] =", learned_rmse)


## Prediction Comparison

What should students observe?
- the learned residual should move the one-step prediction closer to measured data;
- the validation plot matters more than the training fit.


In [ ]:
plot_prediction_eta(
    project_output / "figures" / "prediction_vs_measured_eta.png",
    measured_next[validation],
    nominal_next[validation],
    learned_next_validation,
)
plot_residual_error(
    project_output / "figures" / "residual_prediction_error.png",
    measured_next[validation],
    nominal_next[validation],
    learned_next_validation,
)
plot_rmse(project_output / "figures" / "rmse_comparison.png", nominal_rmse, learned_rmse)


## Replay: Nominal vs Learned Prediction

Run this cell after the simulation to generate a replay.
The replay is saved under outputs/ and is not committed.
Open the generated HTML file in a browser.


In [ ]:
replay_output = repo_root / "outputs" / "project_3_learning_enhanced_prediction"
replay_output.mkdir(parents=True, exist_ok=True)
replay_path = replay_output / "prediction_replay.html"

prediction_stride = max(1, (samples - split) // 30)
save_prediction_replay_html(
    replay_path,
    measured_next=measured_next[validation],
    nominal_next=nominal_next[validation],
    learned_next=learned_next_validation,
    state_index=1,
    stride=prediction_stride,
    interval=120,
)

print(f"Replay saved to: {replay_path}")
display(FileLink(replay_path))


### Interpretation

- Lower one-step RMSE means the learned residual improved this validation set.
- Improvement can be state-dependent; check `xi`, `eta`, and `psi` separately.
- Better prediction is useful for MPC, but it is not a safety guarantee.
- Constraint satisfaction still requires constraints, robustness margins, and validation under closed-loop rollout.


## Modification Cell

Change one value and rerun the fit.

Suggested experiments:
- reduce `modified_samples` to see data scarcity;
- increase `modified_noise_std` to make validation harder;
- set `modified_extrapolation_scale > 1` to test outside the training range.


In [ ]:
# TODO: implement or tune this design choice.
# The surrounding setup is provided so you can focus on the control idea.
